# Part 1 — Table 1 (Judge Selection): Rule-based row

Reproduce **agreement / FPR / FNR** for `StringClassifier` (the rule-based judge)
against the human-majority labels of the 300-prompt JBB judge_comparison split.

The Llama-Guard-3-8B row of Table 1 is computed by `11_appendix_guard3.ipynb`
in a separate kernel (Guard-3 load races with any other vLLM model on a 24 GB GPU
— see PLAN.md KNOWN BUGS). `10_summary.ipynb` joins both rows into one Table 1.

**Outputs**: `results_part1/summary.csv` (1 row: rule-based), `results_part1/raw.json`.


In [ ]:
import sys
sys.path.insert(0, '.')

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


In [ ]:
import os, json
import pandas as pd
from jailbreakbench.classifier import StringClassifier

judge_prompts   = judge_df["prompt"].tolist()
judge_responses = judge_df["target_response"].tolist()

human_majority = judge_df["human_majority"]
if human_majority.dtype == bool:
    ground_truth = human_majority.tolist()
elif human_majority.dtype == object:
    ground_truth = human_majority.str.lower().isin(
        ["unsafe", "yes", "jailbroken", "true"]
    ).tolist()
else:
    ground_truth = human_majority.tolist()
print(f"Ground truth: {sum(ground_truth)} unsafe / "
      f"{len(ground_truth) - sum(ground_truth)} safe")

classifier = StringClassifier('')
rule_preds = classifier.classify_responses(judge_prompts, judge_responses)
metrics_rule = judge_metrics(rule_preds, ground_truth)
print("Rule-based (StringClassifier) results:")
for k, v in metrics_rule.items():
    print(f"  {k}: {v}")


In [ ]:
os.makedirs("results_part1", exist_ok=True)

summary_df = pd.DataFrame([{
    "Classifier": "Rule-based (StringClassifier)",
    "Agreement":  metrics_rule["agreement"],
    "FPR":        metrics_rule["fpr"],
    "FNR":        metrics_rule["fnr"],
}])
summary_df.to_csv("results_part1/summary.csv", index=False)

with open("results_part1/raw.json", "w") as f:
    json.dump({
        "predictions":  [bool(p) for p in rule_preds],
        "ground_truth": [bool(g) for g in ground_truth],
        "metrics":      metrics_rule,
    }, f, indent=2)

print("\n### Table 1 (Part 1 row): Rule-based judge ###")
print(summary_df.to_string(index=False))
